# Exercise 8: South African Market Model

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


In [21]:
lesson_number = 8
print(f'lesson{lesson_number}')

lesson8


## Google Colab Prep

In [22]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if 'lesson_number' not in locals(): lesson_number = 8

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

Not connecting to Google Drive.


In [23]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

Skipping package installation.


In [24]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url = "https://github.com/PriyeshGosai/ich_course_2026/blob/46e109dd68c635d83046eec03d2d2a4afc669366/n-02-za-network-v3.xlsx"
    filename = url.split('/')[-1]  # Extract filename from URL
    
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

Skipping file download.


## Model

In [25]:
start_timestamp = '2026-04-01 00:00:00'
n_days = 7

solver_name = 'gurobi'  # 'gurobi' or 'highs'

unit_commitment = True # True or False for unit commitment of dispatchable generators 

file_name = 'n_03_za_network_v3.xlsx'



In [26]:
import gurobipy as gp
import pypsa
import pandas as pd
import linopy
import plotly.graph_objects as go

import linopy.solvers
linopy.solvers.gurobipy = gp
pypsa.options.api.new_components_api = True

pd.set_option('plotting.backend', 'plotly')

start = pd.Timestamp(start_timestamp)
end = start + pd.Timedelta(days=n_days) - pd.Timedelta(hours=1)

solver_options={
        'TimeLimit': 600,        # 10 minutes
        'MIPGap': 0.02,          # Or accept 2% gap (0.02)
        'LogToConsole': 1 
        }


n = pypsa.Network(file_name)

n.sanitize()

n.set_snapshots(pd.date_range(start, end, freq='h'))

dispatchable_carriers = ['coal','ocgt_diesel','ocgt_avf','sasol_gas'] 
n.generators.static.loc[n.generators.static.carrier.isin(dispatchable_carriers), 
                        'committable'] = unit_commitment

# Cluster the network spatially by bus location, and add load shedding to the clustered network
busmap = n.buses.static['location'].to_dict()  
n_clustered = n.cluster.spatial.cluster_by_busmap(busmap=busmap)

all_busmap = {bus: bus for bus in n.buses.static.index}
n_unconstrained = n.cluster.spatial.cluster_by_busmap(busmap=all_busmap)

n_clustered.optimize(solver_name=solver_name, include_objective_constant = False)
n_unconstrained.optimize(solver_name=solver_name, include_objective_constant = False)


c:\Users\PriyeshGosai\anaconda3\envs\pypsa-model-env\Lib\site-packages\pypsa\network\io.py:1773: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(attrs["default"].to_dict())
c:\Users\PriyeshGosai\anaconda3\envs\pypsa-model-env\Lib\site-packages\pypsa\network\io.py:1773: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(attrs["default"].to_dict())
INFO:pypsa.network.io:Imported network 'pypsa_model' has buses, carriers, generators, global_constraints, lines, links, loads, processes, shapes, shunt_impedances, storage_units, stores, s

Set parameter WLSAccessID


INFO:gurobipy:Set parameter WLSAccessID


Set parameter WLSSecret


INFO:gurobipy:Set parameter WLSSecret


...

INFO:gurobipy:Optimal solution found (tolerance 1.00e-04)


Best objective 2.470047433798e+09, best bound 2.469940384876e+09, gap 0.0043%


INFO:gurobipy:Best objective 2.470047433798e+09, best bound 2.469940384876e+09, gap 0.0043%


INFO:gurobipy:Warning: environment still referenced so free is deferred (Continue to use WLS)
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 124488 primals, 0 duals
Objective: 2.47e+09
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [ ]:
# Extract marginal prices for clustered network
supply_gens_clustered = n_clustered.generators.static[n_clustered.generators.static.type == 'Supply'].index.tolist()
supply_storage_clustered = n_clustered.storage_units.static.index.tolist()

supplier_volume_clustered = pd.concat([
    n_clustered.generators.dynamic.p[supply_gens_clustered],
    n_clustered.storage_units.dynamic.p[supply_storage_clustered].clip(lower=0)
], axis=1)

# Build supplier_prices_clustered efficiently
price_dict_clustered = {}
for gen in supply_gens_clustered:
    if gen in n_clustered.generators.dynamic.marginal_cost.columns:
        price_dict_clustered[gen] = n_clustered.generators.dynamic.marginal_cost[gen]
    else:
        price_dict_clustered[gen] = n_clustered.generators.static.loc[gen, 'marginal_cost']

for storage in supply_storage_clustered:
    if storage in n_clustered.storage_units.dynamic.marginal_cost.columns:
        price_dict_clustered[storage] = n_clustered.storage_units.dynamic.marginal_cost[storage]
    else:
        price_dict_clustered[storage] = n_clustered.storage_units.static.loc[storage, 'marginal_cost']

supplier_prices_clustered = pd.concat(price_dict_clustered, axis=1)
clustered_marginal_prices = supplier_prices_clustered[supplier_volume_clustered > 0].max(axis=1)

# Extract marginal prices for unconstrained network
supply_gens_unconstrained = n_unconstrained.generators.static[n_unconstrained.generators.static.type == 'Supply'].index.tolist()
supply_storage_unconstrained = n_unconstrained.storage_units.static.index.tolist()

supplier_volume_unconstrained = pd.concat([
    n_unconstrained.generators.dynamic.p[supply_gens_unconstrained],
    n_unconstrained.storage_units.dynamic.p[supply_storage_unconstrained].clip(lower=0)
], axis=1)

# Build supplier_prices_unconstrained efficiently
price_dict_unconstrained = {}
for gen in supply_gens_unconstrained:
    if gen in n_unconstrained.generators.dynamic.marginal_cost.columns:
        price_dict_unconstrained[gen] = n_unconstrained.generators.dynamic.marginal_cost[gen]
    else:
        price_dict_unconstrained[gen] = n_unconstrained.generators.static.loc[gen, 'marginal_cost']

for storage in supply_storage_unconstrained:
    if storage in n_unconstrained.storage_units.dynamic.marginal_cost.columns:
        price_dict_unconstrained[storage] = n_unconstrained.storage_units.dynamic.marginal_cost[storage]
    else:
        price_dict_unconstrained[storage] = n_unconstrained.storage_units.static.loc[storage, 'marginal_cost']

supplier_prices_unconstrained = pd.concat(price_dict_unconstrained, axis=1)
unconstrained_marginal_prices = supplier_prices_unconstrained[supplier_volume_unconstrained > 0].max(axis=1)

C:\Users\PriyeshGosai\AppData\Local\Temp\ipykernel_20272\2046089110.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  supplier_prices_clustered[gen] = n_clustered.generators.static.loc[gen, 'marginal_cost']
C:\Users\PriyeshGosai\AppData\Local\Temp\ipykernel_20272\2046089110.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  supplier_prices_clustered[gen] = n_clustered.generators.static.loc[gen, 'marginal_cost']
C:\Users\PriyeshGosai\AppData\Local\Temp\ipykernel_20272\2046089110.py:16: PerformanceWarning: DataFrame is highly

In [28]:
# Create plotly figure
fig = go.Figure()

# Add clustered network trace
fig.add_trace(go.Scatter(
    x=n_clustered.snapshots,
    y=clustered_marginal_prices,
    mode='lines',
    name='Clustered Network',
    line=dict(color='blue', width=2)
))

# Add unconstrained network trace
fig.add_trace(go.Scatter(
    x=n_unconstrained.snapshots,
    y=unconstrained_marginal_prices,
    mode='lines',
    name='Unconstrained Network',
    line=dict(color='red', width=2)
))

# Update layout
fig.update_layout(
    title='Marginal Prices: Clustered vs Unconstrained Network',
    xaxis_title='Time',
    yaxis_title='Marginal Price (ZAR/MWh)',
    hovermode='x unified',
    template='plotly_white',
    height=600,
    width=1200
)

fig.show()

In [29]:
n_clustered.generators.dynamic.p

name,Arnot 1,Arnot 2,Arnot 3,Arnot 4,Arnot 5,Arnot 6,Camden 1,Camden 2,Camden 3,Camden 4,...,Oyster_bay,Perdekraal_east,Roggeveld,Soetwater,Coleskop,Dwarsrug,Phezukomoya,San_Kraal,Waaihoek,Wolf
snapshot,,,,,,,,,,,,,,,,,,,,,
2026-04-01 00:00:00,140.0,140.0,140.0,140.0,140.0,140.0,76.0,76.0,76.0,76.0,...,35.137059,83.483300,79.132630,48.550546,3.336211,31.906129,37.129819,2.181831,16.641477,3.247596
2026-04-01 01:00:00,140.0,140.0,140.0,140.0,140.0,140.0,76.0,76.0,76.0,76.0,...,33.285168,83.396866,74.277467,39.297403,4.405874,22.090566,31.476828,0.581774,12.437549,4.088175
2026-04-01 02:00:00,140.0,140.0,140.0,140.0,140.0,140.0,76.0,76.0,76.0,76.0,...,31.128689,77.806083,68.121859,32.668785,4.220033,14.193776,31.340151,0.458869,8.980569,3.624932
2026-04-01 03:00:00,140.0,140.0,140.0,140.0,140.0,140.0,76.0,76.0,76.0,76.0,...,32.553482,74.205689,70.139460,35.767302,0.661189,9.216945,18.149614,0.605743,9.511841,2.312011
2026-04-01 04:00:00,140.0,140.0,140.0,140.0,140.0,140.0,76.0,76.0,76.0,76.0,...,32.463634,73.400992,70.477842,38.744745,0.000000,10.762413,3.533486,1.173655,10.674629,1.225414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-07 19:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,96.137908,91.114931,106.838603,118.417700,46.573998,33.551541,93.444047,118.073923,65.340545,10.858650
2026-04-07 20:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,85.931047,88.138654,103.807186,115.727837,48.669856,56.771163,94.509836,118.439144,45.668081,3.947249
2026-04-07 21:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,67.351808,83.798643,100.246005,110.785162,52.246634,97.655005,100.233342,118.292310,36.268225,0.697685
